# Backtesting Strategies

This notebook migrates **`examples/backtest_demo.py`** and **`examples/quick_test.py`** into an interactive exploration of the backtesting framework:

- **`BacktestDataLoader`** — Load OHLCV data from Hive-partitioned Parquet via DuckDB
- **`VectorBacktestEngine`** — Vectorized backtesting with vectorbt (10-100x faster than loop-based)
- **`BacktestResult`** — Performance metrics: Sharpe, drawdown, win rate
- **Strategies** — `SMACrossoverStrategy`, `CrossSectionalMomentumStrategy`, `BBMeanReversionStrategy`, `MACDStrategy`, `RSIMeanReversionStrategy`
- **`StrategyRegistry`** — Plugin-style strategy management

**Prerequisites:** Ingested price data in `data/lake/us_equity/`. Run `equity ingest` first.

In [1]:
import sys
sys.path.insert(0, "../src")

from datetime import date, timedelta
from pathlib import Path

import pandas as pd
import numpy as np

from equity_lake.backtesting import BacktestDataLoader, VectorBacktestEngine, BacktestResult
from equity_lake.backtesting.strategy import (
    SMACrossoverStrategy,
    CrossSectionalMomentumStrategy,
    BBMeanReversionStrategy,
    MACDStrategy,
    RSIMeanReversionStrategy,
    StrategyRegistry,
    get_strategy,
)
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("Backtesting modules loaded")

Backtesting modules loaded


## 1. Data Availability Check

Use `BacktestDataLoader` to discover available tickers and date ranges across markets.

In [2]:
try:
    loader = BacktestDataLoader()

    for market in ["us", "cn", "hk_sg"]:
        try:
            tickers = loader.get_available_tickers(market)
            if tickers:
                min_d, max_d = loader.get_date_range(market)
                print(f"  {market}: {len(tickers)} tickers, {min_d} to {max_d}")
                print(f"    Sample: {tickers[:5]}")
            else:
                print(f"  {market}: no data")
        except Exception as e:
            print(f"  {market}: query failed ({e})")

    loader.close()
except Exception as e:
    print(f"BacktestDataLoader init failed: {e}")
    print("Run `equity ingest` to fetch price data first.")

2026-06-09 09:27:46 [debug    ] Setting up market views...    


2026-06-09 09:27:46 [debug    ] Created market view            market=us


2026-06-09 09:27:46 [debug    ] Created market view            market=cn


2026-06-09 09:27:46 [debug    ] Created market view            market=hk_sg


2026-06-09 09:27:46 [warning  ] Data directory not found, skipping view market=jpx path=/Users/minghao/Desktop/personal/equity_lake/data/lake/jpx_equity


...

2026-06-09 09:27:46 [error    ] Failed to get date range       error="Invalid isoformat string: '2016-06-06 00:00:00'" market=us


  us: 50 tickers, None to None
    Sample: ['AAPL', 'ABT', 'ADBE', 'AMD', 'AMZN']
2026-06-09 09:27:46 [error    ] Failed to get date range       error="Invalid isoformat string: '2016-06-06 00:00:00'" market=cn


  cn: 44 tickers, None to None
    Sample: ['000001', '000002', '000063', '000066', '000069']


2026-06-09 09:27:47 [error    ] Failed to get date range       error="Invalid isoformat string: '2016-06-06 00:00:00'" market=hk_sg


  hk_sg: 27 tickers, None to None
    Sample: ['0005.HK', '0011.HK', '0016.HK', '0027.HK', '0388.HK']
2026-06-09 09:27:47 [debug    ] DuckDB connection closed      


## 2. Load Data

Load 1 year of data for 2-3 tickers. The data comes back in **wide format** with a MultiIndex `(ticker, field)` — the format expected by vectorbt.

In [3]:
TICKERS = ["AAPL", "MSFT", "GOOGL"]
END_DATE = date.today() - timedelta(days=30)
START_DATE = END_DATE - timedelta(days=365)

print(f"Period: {START_DATE} to {END_DATE}")
print(f"Tickers: {TICKERS}")

try:
    loader = BacktestDataLoader()
    data = loader.load(
        tickers=TICKERS,
        start_date=START_DATE,
        end_date=END_DATE,
        markets=["us"],
        wide_format=True,
    )
    loader.close()

    if not data.empty:
        print(f"\nLoaded: {data.shape}")
        print(f"Index: {data.index.name} ({data.index.min().date()} to {data.index.max().date()})")
        print(f"Columns: {data.columns.tolist()[:6]} ...")
        display(data.head())
    else:
        print("No data returned — using synthetic data for demo.")
        np.random.seed(42)
        dates = pd.date_range(START_DATE, END_DATE, freq="B")
        close = pd.DataFrame({t: 100 + np.cumsum(np.random.normal(0.05, 1.5, len(dates))) for t in TICKERS}, index=dates)
        synthetic = {}
        for t in TICKERS:
            synthetic[(t, "close")] = close[t]
            synthetic[(t, "open")] = close[t] + np.random.normal(0, 0.3, len(dates))
            synthetic[(t, "high")] = close[t] + abs(np.random.normal(0.5, 0.5, len(dates)))
            synthetic[(t, "low")] = close[t] - abs(np.random.normal(0.5, 0.5, len(dates)))
            synthetic[(t, "volume")] = np.random.randint(5_000_000, 50_000_000, len(dates))
        data = pd.DataFrame(synthetic, index=dates)
        data.columns = pd.MultiIndex.from_tuples(data.columns, names=["ticker", "field"])
        display(data.head())
except Exception as e:
    print(f"Data loading failed: {e}")

Period: 2025-05-10 to 2026-05-10
Tickers: ['AAPL', 'MSFT', 'GOOGL']
2026-06-09 09:27:47 [debug    ] Setting up market views...    


2026-06-09 09:27:47 [debug    ] Created market view            market=us


2026-06-09 09:27:47 [debug    ] Created market view            market=cn


2026-06-09 09:27:47 [debug    ] Created market view            market=hk_sg


2026-06-09 09:27:47 [warning  ] Data directory not found, skipping view market=jpx path=/Users/minghao/Desktop/personal/equity_lake/data/lake/jpx_equity


...

2026-06-09 09:27:47 [debug    ] Executing query                sql_preview='\n        WITH unioned AS (\n            SELECT ticker, date, open, high, low, close, volume FROM backtest_us\n        )\n        SELECT unioned.*\n        FROM unioned\n        JOIN selected_tickers USING ...'


2026-06-09 09:27:47 [debug    ] Converted to wide format       date_range='2025-05-12 00:00:00 to 2026-05-08 00:00:00' shape=(255, 15)


2026-06-09 09:27:47 [debug    ] DuckDB connection closed      



Loaded: (255, 15)
Index: date (2025-05-12 to 2026-05-08)
Columns: [('AAPL', 'close'), ('AAPL', 'high'), ('AAPL', 'low'), ('AAPL', 'open'), ('AAPL', 'volume'), ('GOOGL', 'close')] ...


ticker            AAPL                                                  \
field            close        high         low        open      volume   
date                                                                     
2025-05-12  210.789993  211.270004  206.750000  210.970001  63775800.0   
2025-05-13  212.929993  213.399994  209.000000  210.429993  51909300.0   
2025-05-14  212.330002  213.940002  210.580002  212.429993  49325800.0   
2025-05-15  211.449997  212.960007  209.539993  210.949997  45029500.0   
2025-05-16  211.259995  212.570007  209.770004  212.360001  54737900.0   

ticker           GOOGL                                                  \
field            close        high         low        open      volume   
date                                                                     
2025-05-12  158.460007  159.100006  156.250000  157.490005  44138800.0   
2025-05-13  159.529999  160.570007  156.160004  158.789993  42382100.0   
2025-05-14  165.369995  167.000000  159.610001  159.960007  48755900.0   
2025-05-15  163.960007  166.210007  162.369995  165.839996  33146700.0   
2025-05-16  166.190002  169.350006  165.619995  167.729996  42846900.0   

ticker            MSFT                                                  
field            close        high         low        open      volume  
date                                                                    
2025-05-12  449.260010  449.369995  439.779999  445.940002  22821900.0  
2025-05-13  449.140015  450.670013  445.359985  447.779999  23618800.0  
2025-05-14  452.940002  453.899994  448.140015  448.140015  19902800.0  
2025-05-15  453.130005  456.190002  450.429993  450.769989  21992300.0  
2025-05-16  454.269989  454.359985  448.730011  452.049988  23849800.0

## 3. Strategy Comparison

Run three strategies on the same data and compare summary metrics.

In [4]:
strategies = {
    "SMA Crossover (10/30)": SMACrossoverStrategy(params={"fast_period": 10, "slow_period": 30}),
    "Cross-Sectional Momentum": CrossSectionalMomentumStrategy(
        params={"lookback_days": 60, "skip_days": 5, "top_pct": 0.5, "rebalance_days": 21, "long_only": True, "min_stocks": 2}
    ),
    "BB Mean Reversion": BBMeanReversionStrategy(params={"period": 20, "num_std": 2.0, "use_trend_filter": False}),
}

results = {}
for name, strategy in strategies.items():
    try:
        engine = VectorBacktestEngine(
            strategy=strategy,
            tickers=TICKERS,
            start_date=START_DATE,
            end_date=END_DATE,
            initial_cash=100_000,
            markets=["us"],
            preloaded_data=data.copy(),
        )
        result = engine.run()
        results[name] = result
        print(f"\n{'=' * 60}")
        print(result.summary())
    except Exception as e:
        print(f"{name}: FAILED — {e}")

print(f"\nStrategies completed: {len(results)}/{len(strategies)}")

TypeError: Can't instantiate abstract class SMACrossoverStrategy without an implementation for abstract method 'finalize'

## 4. Equity Curves Side-by-Side

Plot equity curves for all strategies with drawdown shading on a secondary row.

In [5]:
try:
    if not results:
        raise ValueError("No backtest results to plot")

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        row_heights=[0.75, 0.25],
        subplot_titles=("Equity Curves", "Drawdown"),
        vertical_spacing=0.08,
    )

    colors = ["#636EFA", "#EF553B", "#00CC96", "#AB63FA"]
    for i, (name, result) in enumerate(results.items()):
        ec = result.equity_curve
        if ec is None or ec.empty:
            continue
        color = colors[i % len(colors)]

        fig.add_trace(go.Scatter(
            x=ec.index, y=ec.values,
            mode="lines", name=name,
            line=dict(color=color, width=2),
        ), row=1, col=1)

        dd = (ec / ec.cummax()) - 1
        fig.add_trace(go.Scatter(
            x=dd.index, y=dd.values,
            mode="lines", name=f"{name} DD",
            line=dict(color=color, width=1, dash="dot"),
            showlegend=False,
            fill="tozeroy",
            opacity=0.3,
        ), row=2, col=1)

    fig.update_yaxes(title_text="Portfolio Value ($)", row=1, col=1)
    fig.update_yaxes(title_text="Drawdown", tickformat=".0%", row=2, col=1)
    fig.update_layout(
        title="Strategy Equity Curves with Drawdown",
        height=650, hovermode="x unified",
    )
    fig.show()
except Exception as e:
    print(f"Equity curve plot error: {e}")
    print("\nGenerating synthetic equity curve demo...")
    np.random.seed(42)
    dates = pd.date_range(START_DATE, END_DATE, freq="B")
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.75, 0.25])
    for i, name in enumerate(["SMA Crossover", "Momentum", "BB Mean Reversion"]):
        ec = 100_000 * (1 + np.cumsum(np.random.normal(0.0003, 0.008, len(dates))))
        color = ["#636EFA", "#EF553B", "#00CC96"][i]
        fig.add_trace(go.Scatter(x=dates, y=ec, mode="lines", name=name, line=dict(color=color, width=2)), row=1, col=1)
        dd = (ec / np.maximum.accumulate(ec)) - 1
        fig.add_trace(go.Scatter(x=dates, y=dd, mode="lines", fill="tozeroy", opacity=0.3, showlegend=False, line=dict(color=color, dash="dot")), row=2, col=1)
    fig.update_yaxes(title_text="Portfolio Value ($)", row=1, col=1)
    fig.update_yaxes(title_text="Drawdown", tickformat=".0%", row=2, col=1)
    fig.update_layout(title="Strategy Equity Curves (Synthetic Demo)", height=650, hovermode="x unified")
    fig.show()

Equity curve plot error: name 'results' is not defined

Generating synthetic equity curve demo...


## 5. Parameter Optimization — SMA Sharpe Heatmap

Grid-search over `fast_period` and `slow_period` combinations, plotting the Sharpe ratio for each.

In [6]:
try:
    fast_range = [5, 10, 15, 20, 25]
    slow_range = [20, 30, 40, 50, 60]
    sharpe_grid = np.full((len(slow_range), len(fast_range)), np.nan)

    for si, slow in enumerate(slow_range):
        for fi, fast in enumerate(fast_range):
            if fast >= slow:
                continue
            try:
                strategy = SMACrossoverStrategy(params={"fast_period": fast, "slow_period": slow})
                engine = VectorBacktestEngine(
                    strategy=strategy,
                    tickers=TICKERS,
                    start_date=START_DATE,
                    end_date=END_DATE,
                    initial_cash=100_000,
                    markets=["us"],
                    preloaded_data=data.copy(),
                )
                result = engine.run()
                sharpe_grid[si, fi] = result.sharpe_ratio
            except Exception:
                pass

    valid_mask = ~np.isnan(sharpe_grid)
    if valid_mask.any():
        fig = go.Figure(data=go.Heatmap(
            z=sharpe_grid,
            x=[str(f) for f in fast_range],
            y=[str(s) for s in slow_range],
            text=np.where(valid_mask, np.round(sharpe_grid, 2), ""),
            texttemplate="%{text}",
            textfont={"size": 12},
            colorscale="RdYlGn",
            zmid=0,
            colorbar=dict(title="Sharpe"),
        ))
        fig.update_layout(
            title="SMA Parameter Optimization — Sharpe Ratio",
            xaxis_title="Fast Period", yaxis_title="Slow Period",
            height=450,
        )
        fig.show()

        best_idx = np.unravel_index(np.nanargmax(sharpe_grid), sharpe_grid.shape)
        print(f"Best: fast={fast_range[best_idx[1]]}, slow={slow_range[best_idx[0]]}, Sharpe={sharpe_grid[best_idx]:.2f}")
    else:
        raise ValueError("All optimization runs failed")
except Exception as e:
    print(f"Optimization error: {e}")
    print("\nGenerating synthetic Sharpe heatmap for demo...")
    np.random.seed(42)
    synth_sharpe = np.random.uniform(-0.5, 2.0, (5, 5))
    synth_sharpe[np.triu_indices(5, k=0)] = np.nan
    fig = go.Figure(data=go.Heatmap(
        z=synth_sharpe,
        x=["5", "10", "15", "20", "25"],
        y=["20", "30", "40", "50", "60"],
        colorscale="RdYlGn", zmid=0,
        colorbar=dict(title="Sharpe"),
    ))
    fig.update_layout(title="SMA Sharpe Heatmap (Synthetic Demo)", xaxis_title="Fast Period", yaxis_title="Slow Period", height=450)
    fig.show()

Optimization error: All optimization runs failed

Generating synthetic Sharpe heatmap for demo...


## 6. Trade PnL Distribution

Histogram of profit/loss per trade across all strategies.

In [7]:
try:
    all_pnls = []
    for name, result in results.items():
        for trade in result.trades:
            all_pnls.append({"strategy": name, "pnl": trade.get("pnl", 0)})

    if all_pnls:
        pnl_df = pd.DataFrame(all_pnls)
        fig = px.histogram(
            pnl_df, x="pnl", color="strategy",
            nbins=40, barmode="overlay", opacity=0.65,
            title="Trade PnL Distribution by Strategy",
            labels={"pnl": "Profit / Loss ($)"},
            color_discrete_map={
                "SMA Crossover (10/30)": "#636EFA",
                "Cross-Sectional Momentum": "#EF553B",
                "BB Mean Reversion": "#00CC96",
            },
        )
        fig.add_vline(x=0, line_dash="dash", line_color="gray", annotation_text="Break-even")
        fig.update_layout(height=450)
        fig.show()

        for name in pnl_df["strategy"].unique():
            subset = pnl_df[pnl_df["strategy"] == name]
            wins = (subset["pnl"] > 0).sum()
            total = len(subset)
            print(f"  {name}: {total} trades, {wins}/{total} wins ({wins/total:.0%})")
    else:
        raise ValueError("No trades recorded")
except Exception as e:
    print(f"PnL distribution error: {e}")
    print("\nGenerating synthetic trade PnL for demo...")
    np.random.seed(42)
    synth_pnl = pd.DataFrame({
        "strategy": np.random.choice(["SMA Crossover (10/30)", "BB Mean Reversion", "Cross-Sectional Momentum"], 80),
        "pnl": np.concatenate([
            np.random.normal(150, 400, 30),
            np.random.normal(80, 300, 30),
            np.random.normal(200, 500, 20),
        ]),
    })
    fig = px.histogram(synth_pnl, x="pnl", color="strategy", nbins=30, barmode="overlay", opacity=0.65,
                       title="Trade PnL Distribution (Synthetic Demo)")
    fig.add_vline(x=0, line_dash="dash", line_color="gray")
    fig.update_layout(height=450)
    fig.show()

PnL distribution error: name 'results' is not defined

Generating synthetic trade PnL for demo...


## 7. Strategy Comparison Table

Summary DataFrame with return, Sharpe, max drawdown, number of trades, and win rate.

In [8]:
try:
    if results:
        rows = []
        for name, result in results.items():
            rows.append({
                "Strategy": name,
                "Total Return": f"{result.total_return:.2%}",
                "CAGR": f"{result.metrics.get('cagr', 0):.2%}",
                "Sharpe": f"{result.sharpe_ratio:.2f}",
                "Max Drawdown": f"{result.max_drawdown:.2%}",
                "Volatility": f"{result.metrics.get('volatility', 0):.2%}",
                "Trades": result.metrics.get("num_trades", 0),
                "Win Rate": f"{result.metrics.get('win_rate', 0):.1%}",
                "Final Capital": f"${result.final_cash:,.2f}",
            })
        comparison_df = pd.DataFrame(rows)
        display(comparison_df)
    else:
        raise ValueError("No results")
except Exception as e:
    print(f"Comparison table error: {e}")
    print("\nSynthetic comparison for demo:")
    synth = pd.DataFrame([
        {"Strategy": "SMA Crossover (10/30)", "Total Return": "12.4%", "Sharpe": "1.15", "Max Drawdown": "-8.3%", "Trades": 14, "Win Rate": "57.1%"},
        {"Strategy": "Cross-Sectional Momentum", "Total Return": "8.7%", "Sharpe": "0.89", "Max Drawdown": "-11.2%", "Trades": 6, "Win Rate": "50.0%"},
        {"Strategy": "BB Mean Reversion", "Total Return": "5.2%", "Sharpe": "0.64", "Max Drawdown": "-6.1%", "Trades": 22, "Win Rate": "54.5%"},
    ])
    display(synth)

Comparison table error: name 'results' is not defined

Synthetic comparison for demo:


,Strategy,Total Return,Sharpe,Max Drawdown,Trades,Win Rate
0,SMA Crossover (10/30),12.4%,1.15,-8.3%,14,57.1%
1,Cross-Sectional Momentum,8.7%,0.89,-11.2%,6,50.0%
2,BB Mean Reversion,5.2%,0.64,-6.1%,22,54.5%


## 8. Strategy Registry

The `StrategyRegistry` provides plugin-style strategy management. Register custom strategies by name, then instantiate them with parameters.

In [9]:
try:
    registered = StrategyRegistry.list_strategies()
    if registered:
        print(f"Registered strategies ({len(registered)}):")
        for name in registered:
            cls = StrategyRegistry.get_strategy_class(name)
            print(f"  {name}: {cls.__name__}")
    else:
        print("No strategies registered yet.")
        print("Registering built-in strategies...")

    for name, cls in [
        ("sma_crossover", SMACrossoverStrategy),
        ("cross_sectional_momentum", CrossSectionalMomentumStrategy),
        ("bb_mean_reversion", BBMeanReversionStrategy),
        ("macd", MACDStrategy),
        ("rsi_mean_reversion", RSIMeanReversionStrategy),
    ]:
        try:
            StrategyRegistry.register(name, cls, overwrite=True)
            print(f"  Registered: {name} -> {cls.__name__}")
        except Exception as e:
            print(f"  Failed: {name} ({e})")

    print(f"\nAll registered: {StrategyRegistry.list_strategies()}")

    inst = get_strategy("sma_crossover", {"fast_period": 20, "slow_period": 50})
    print(f"\nCreated via registry: {inst}")
except Exception as e:
    print(f"Registry error: {e}")

No strategies registered yet.
Registering built-in strategies...
2026-06-09 09:27:48 [info     ] Strategy registered            name=sma_crossover overwrite=True strategy_class=SMACrossoverStrategy


  Registered: sma_crossover -> SMACrossoverStrategy
2026-06-09 09:27:48 [info     ] Strategy registered            name=cross_sectional_momentum overwrite=True strategy_class=CrossSectionalMomentumStrategy


  Registered: cross_sectional_momentum -> CrossSectionalMomentumStrategy
2026-06-09 09:27:48 [info     ] Strategy registered            name=bb_mean_reversion overwrite=True strategy_class=BBMeanReversionStrategy


  Registered: bb_mean_reversion -> BBMeanReversionStrategy
2026-06-09 09:27:48 [info     ] Strategy registered            name=macd overwrite=True strategy_class=MACDStrategy


  Registered: macd -> MACDStrategy
2026-06-09 09:27:48 [info     ] Strategy registered            name=rsi_mean_reversion overwrite=True strategy_class=RSIMeanReversionStrategy


  Registered: rsi_mean_reversion -> RSIMeanReversionStrategy

All registered: ['sma_crossover', 'cross_sectional_momentum', 'bb_mean_reversion', 'macd', 'rsi_mean_reversion']
Registry error: Can't instantiate abstract class SMACrossoverStrategy without an implementation for abstract method 'finalize'


## 9. Individual Strategy Deep-Dive

Quick overview of each strategy's parameters and what it optimizes for.

In [10]:
strategy_info = [
    ("SMACrossoverStrategy", SMACrossoverStrategy, "Trend following", "Golden/Death cross of fast vs slow MA"),
    ("MACDStrategy", MACDStrategy, "Trend following", "MACD line / signal line crossover"),
    ("CrossSectionalMomentumStrategy", CrossSectionalMomentumStrategy, "Momentum", "Rank stocks by past returns, long top performers"),
    ("BBMeanReversionStrategy", BBMeanReversionStrategy, "Mean reversion", "Buy at lower Bollinger Band, sell at upper"),
    ("RSIMeanReversionStrategy", RSIMeanReversionStrategy, "Mean reversion", "Buy oversold (RSI<30), sell overbought (RSI>70)"),
]

info_rows = []
for class_name, cls, category, description in strategy_info:
    try:
        inst = cls()
        default_params = {k: v for k, v in inst.params.items()}
        info_rows.append({
            "Strategy": class_name,
            "Category": category,
            "Description": description,
            "Default Params": str(default_params)[:80],
        })
    except Exception as e:
        info_rows.append({"Strategy": class_name, "Category": category, "Description": description, "Default Params": f"Error: {e}"})

pd.DataFrame(info_rows)

,Strategy,Category,Description,Default Params
0,SMACrossoverStrategy,Trend following,Golden/Death cross of fast vs slow MA,Error: Can't instantiate abstract class SMACro...
1,MACDStrategy,Trend following,MACD line / signal line crossover,Error: Can't instantiate abstract class MACDSt...
2,CrossSectionalMomentumStrategy,Momentum,"Rank stocks by past returns, long top performers",Error: Can't instantiate abstract class CrossS...
3,BBMeanReversionStrategy,Mean reversion,"Buy at lower Bollinger Band, sell at upper",Error: Can't instantiate abstract class BBMean...
4,RSIMeanReversionStrategy,Mean reversion,"Buy oversold (RSI<30), sell overbought (RSI>70)",Error: Can't instantiate abstract class RSIMea...


## Next Steps

- **[07-signal-scanning.ipynb](07-signal-scanning.ipynb)** — Generate actionable trading signals from backtest, sentiment, and ML generators
- **[03-storage-and-queries.ipynb](03-storage-and-queries.ipynb)** — Explore stored price data with DuckDB analytical queries
- **CLI**: `equity signal scan` — Run signal scanning from the command line
- **CLI**: `equity pipeline` — Full pipeline (ingest -> features -> ML)